In [10]:
import math
import torch
import numpy as np
from torch import nn, optim

In [11]:
#Sigma Sampler function(log-uniform)
class sigma_sampler_loguniform:
    #initialize
    def __init__(self, sigma_min=0.002, sigma_max=80):
        #initialize relative parameters
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.log_min = math.log(sigma_min)
        self.log_max = math.log(sigma_max)
    def sample(self, batch_size, device):
        #sample log space
        u = torch.rand(batch_size, device=device)
        log_sigma = self.log_min + (self.log_max - self.log_min) * u
        #project to sigma
        sigma = torch.exp(log_sigma)
        return sigma

In [12]:
#Sigma Sampler function(log-normal)
def sigma_sampler_lognormal(batch_size):
    #mean and std value of sigma
    mean = -1.2
    std = 1.2
    sigma = torch.exp(torch.randn(batch_size) * std + mean)
    return sigma.clamp(0.002, 80)

In [13]:
#Sigma Sampler function(quantile)
class sigma_sampler_quantile:
    #initialize
    def __init__(self, sigma_min=0.002, sigma_max=80, mu=-1.2, std=1.2, num_buckets=8):
        #initialize relative parameters
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.mu = mu
        self.std = std
        self.num_buckets = num_buckets
        #log space boundaries
        self.log_min = np.log(sigma_min)
        self.log_max = np.log(sigma_max)
        #truncated CDF bounds
        self.cdf_min = self._normal_cdf(self.log_min)
        self.cdf_max = self._normal_cdf(self.log_max)
    def _normal_cdf(self, x):
        return 0.5 * (1 + math.erf((x - self.mu) / (self.std * math.sqrt(2))))
    def _normal_icdf(self, u):
        return self.mu + self.std * math.sqrt(2) * torch.erfinv(2 * u - 1)
    #forward propagation
    def sample(self, batch_size, device):
        #random select bucket index
        bucket_idx = torch.randint(low=0, high=self.num_buckets, size=(batch_size,), device=device)
        #select boundaries
        q_low = bucket_idx / self.num_buckets
        q_high = (bucket_idx + 1) / self.num_buckets
        #uniform sample in bucket
        u = torch.rand(batch_size, device=device)
        u = q_low + (q_high - q_low) * u
        #project to truncated CDF range
        u = self.cdf_min + u * (self.cdf_max - self.cdf_min)
        #inverse to log sigma
        log_sigma = self._normal_icdf(u)
        sigma = torch.exp(log_sigma)
        return sigma, bucket_idx